# Latent Fair Value Estimation via Adaptive Kalman Filtering

## Market Microstructure Motivation

In modern electronic limit order books, the observed execution "price" is rarely the true *fair value* of an asset. Microstructure noise—driven by bid-ask bounce, transient order flow imbalances, and discrete tick sizes—causes the observed price to constantly oscillate around a latent (unobservable) fair value.

If we assume the latent fair value $\mu_t$ follows a random walk with drift, and the observed price $y_t$ is a noisy manifestation of $\mu_t$, we can formulate this as a Linear State-Space Model. A standard Exponentially Weighted Moving Average (EWMA) is slow to react to sudden regime shifts. By contrast, a Kalman Filter explicitly models the uncertainty (variance) of the fair value and can adaptively update its internal noise matrices via Expectation-Maximization (EM) techniques when market conditions change.


## Mathematical Derivation

Let our state vector be $x_t = [\mu_t, \delta_t]^T$, where $\mu_t$ is fair value and $\delta_t$ is the drift.

**1. Predict Step**
The state transitions according to a constant velocity kinematic model:
$$x_{t|t-1} = F x_{t-1|t-1} + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$
$$P_{t|t-1} = F P_{t-1|t-1} F^T + Q$$
Where $F = \begin{bmatrix} 1 & dt \\ 0 & 1 \end{bmatrix}$.

**2. Update Step**
We observe the noisy price $y_t$:
$$y_t = H x_{t} + v_t, \quad v_t \sim \mathcal{N}(0, R)$$
Where $H = \begin{bmatrix} 1 & 0 \end{bmatrix}$.

The innovation (residual) is $v_t = y_t - H x_{t|t-1}$.
The innovation covariance is $S_t = H P_{t|t-1} H^T + R$.
The Kalman Gain determines how much we trust the observation vs. our prediction:
$$K_t = P_{t|t-1} H^T S_t^{-1}$$

We update our posterior state and covariance:
$$x_{t|t} = x_{t|t-1} + K_t v_t$$
$$P_{t|t} = (I - K_t H) P_{t|t-1}$$

### Adaptive Q and R Estimation
In an adaptive setting, we recursively update the observation noise $R$ and process noise $Q$ using a Robbins-Monro style online EM update with a forgetting factor $\alpha$:
$$R_{t} = (1 - \alpha) R_{t-1} + \alpha (v_t^2 - H P_{t|t-1} H^T)$$
$$Q_{t} = (1 - \alpha) Q_{t-1} + \alpha (K_t v_t v_t^T K_t^T)$$


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from benchmark_kalman import run_benchmark

# Run the benchmark comparing Adaptive Kalman against EWMA and SMA
run_benchmark()
